# Load Inventory Minifigs from Rebrickable CSV Download
Downloads `inventory_minifigs.csv.gz` from the Rebrickable bulk download page and loads it into a Spark DataFrame.

In [ ]:
import io
import gzip
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

DOWNLOAD_URL = "https://cdn.rebrickable.com/media/downloads/inventory_minifigs.csv.gz"

In [ ]:
response = requests.get(DOWNLOAD_URL, timeout=60)
response.raise_for_status()

with gzip.open(io.BytesIO(response.content)) as f:
    df_pd = pd.read_csv(f)

print(f"Downloaded {len(df_pd)} rows")
print(df_pd.dtypes)
df_pd.head()

In [ ]:
schema = StructType([
    StructField("inventory_id", IntegerType(), nullable=False),
    StructField("fig_num",      StringType(),  nullable=False),
    StructField("quantity",     IntegerType(), nullable=False),
])

rows = [
    (
        int(row["inventory_id"]),
        str(row["fig_num"]),
        int(row["quantity"]),
    )
    for _, row in df_pd.iterrows()
]

spark = SparkSession.builder.getOrCreate()
df_inventory_minifigs = spark.createDataFrame(rows, schema=schema)

df_inventory_minifigs.printSchema()
df_inventory_minifigs.display(10, truncate=False)

In [ ]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

# Generate timestamp components for partitioned path and filename
now = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_inventory_minifigs_out = (
    df_inventory_minifigs
    .withColumn("_load_datetime", current_timestamp())
    .withColumn("_record_source", lit("CSV_DOWNLOAD"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/inventory_minifigs"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_inventory_minifigs_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_inventory_minifigs_out.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Inventory minifigs written to: {FINAL_PATH}")

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import TimestampType

DELTA_TABLE_PATH = "/Volumes/lego_brickbase/bronze/delta_volume/inventory_minifigs"

# Natural key and tracked business columns
KEY_COLS     = ["inventory_id", "fig_num"]
TRACKED_COLS = ["quantity"]

# Read the Parquet file written in the previous step
df_source = spark.read.parquet(FINAL_PATH)

# Attach SCD2 metadata columns
df_incoming = (
    df_source
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to",   lit(None).cast(TimestampType()))
    .withColumn("is_current", lit(True))
    .withColumn("is_deleted", lit(False))
)

if not DeltaTable.isDeltaTable(spark, DELTA_TABLE_PATH):
    # First load — write all records as current
    df_incoming.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    print(f"Delta table created at {DELTA_TABLE_PATH}")
else:
    delta_table    = DeltaTable.forPath(spark, DELTA_TABLE_PATH)
    join_condition = " AND ".join([f"tgt.{c} = src.{c}" for c in KEY_COLS])
    change_cond    = " OR ".join([f"tgt.{c} != src.{c}" for c in TRACKED_COLS])

    # Step 1: Expire changed records; soft-delete records missing from source
    (
        delta_table.alias("tgt")
        .merge(df_incoming.alias("src"), f"{join_condition} AND tgt.is_current = true")
        .whenMatchedUpdate(
            condition=change_cond,
            set={"valid_to": "src.valid_from", "is_current": "false"}
        )
        .whenNotMatchedBySourceUpdate(
            condition="tgt.is_current = true",
            set={"valid_to": "current_timestamp()", "is_current": "false", "is_deleted": "true"}
        )
        .execute()
    )

    # Step 2: Insert new records and new versions of changed records
    df_current   = delta_table.toDF().filter("is_current = true")
    df_to_insert = df_incoming.join(df_current.select(*KEY_COLS), on=KEY_COLS, how="left_anti")
    df_to_insert.write.format("delta").mode("append").save(DELTA_TABLE_PATH)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).filter("is_current = true").count()
print(f"Active rows in Delta table: {row_count:,}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS lego_brickbase.bronze.raw_inventory_minifigs
AS SELECT * FROM delta.`/Volumes/lego_brickbase/bronze/delta_volume/inventory_minifigs`;